##  Business Problem: Predicting Customer Churn Using Neural Networks for Proactive Retention

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")
import tensorflow as tf
import keras

## Data Understanding

In [2]:
df=pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [4]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [5]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [6]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [7]:
df.duplicated().sum()

0

In [8]:
df['RowNumber'].nunique()

10000

In [9]:
df['CustomerId'].nunique()

10000

In [10]:
df['Surname'].nunique()

2932

**RowNumber & CustomerId are unique columns thus, they won't decide the output, so we can drop them as they are irrevalent**

In [11]:
df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)
df.head(3)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [12]:
df['Tenure'].nunique()

11

In [13]:
df['Age'].nunique()

70

In [14]:
df.keys()

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')

In [15]:
continuous=['CreditScore','Balance','EstimatedSalary']
discrete_count=['Age','Tenure','NumOfProducts', 'HasCrCard', 'IsActiveMember','Exited']
discrete_categorical=['Geography','Gender']

In [16]:
df[continuous].describe()

,CreditScore,Balance,EstimatedSalary
count,10000.000000,10000.000000,10000.000000
mean,650.528800,76485.889288,100090.239881
std,96.653299,62397.405202,57510.492818
min,350.000000,0.000000,11.580000
25%,584.000000,0.000000,51002.110000
50%,652.000000,97198.540000,100193.915000
75%,718.000000,127644.240000,149388.247500
max,850.000000,250898.090000,199992.480000


In [17]:
df[discrete_count].describe()

,Age,Tenure,NumOfProducts,HasCrCard,IsActiveMember,Exited
count,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000
mean,38.921800,5.012800,1.530200,0.70550,0.515100,0.203700
std,10.487806,2.892174,0.581654,0.45584,0.499797,0.402769
min,18.000000,0.000000,1.000000,0.00000,0.000000,0.000000
25%,32.000000,3.000000,1.000000,0.00000,0.000000,0.000000
50%,37.000000,5.000000,1.000000,1.00000,1.000000,0.000000
75%,44.000000,7.000000,2.000000,1.00000,1.000000,0.000000
max,92.000000,10.000000,4.000000,1.00000,1.000000,1.000000


In [18]:
df[discrete_categorical].describe()

,Geography,Gender
count,10000,10000
unique,3,2
top,France,Male
freq,5014,5457


In [19]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

## Nominal Encoding

In [20]:
df=pd.get_dummies(df,drop_first=True)
df.head(3)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False


In [21]:
X=df.drop('Exited',axis=1)
y=df['Exited']
X.head(3)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,False,False,False


In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)

In [23]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.fit_transform(X_test)

In [24]:
X_train.shape

(8000, 11)

## Modelling

In [25]:
from keras.models import Sequential
ann_model=Sequential()

In [26]:
from keras.layers import Dense
ann_model.add(Dense(input_dim=11,units=6,kernel_initializer='uniform',activation='relu'))

In [27]:
ann_model.add(Dense(units=6,kernel_initializer='uniform',activation='relu'))

In [28]:
ann_model.add(Dense(units=1,kernel_initializer='uniform',activation='sigmoid'))

In [29]:
ann_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [30]:
ann_model.fit(X_train,y_train,batch_size=32,epochs=20)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7969 - loss: 0.6370  
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7989 - loss: 0.4348  
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8029 - loss: 0.4234  
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - accuracy: 0.8020 - loss: 0.4211
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 939us/step - accuracy: 0.8205 - loss: 0.4123
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8246 - loss: 0.4172  
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8329 - loss: 0.4056
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8467 - loss: 0.3992
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 998us/step - accuracy: 0.8291 - loss: 0.4136
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8347 - loss: 0.4112
Epoch 11/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - accuracy: 0.8467 - loss: 0.3935
Epoch 12/20
250/250 ━━━━━━━━━━━━━━━━

In [31]:
y_pred=ann_model.predict(X_test)
y_pred=(y_pred>0.5)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [32]:
from sklearn.metrics import confusion_matrix,accuracy_score
print(f"Test accuracy is {accuracy_score(y_test,y_pred)}")
confusion_matrix(y_test,y_pred)

Test accuracy is 0.8425


array([[1555,   40],
       [ 275,  130]], dtype=int64)

### Cross Validation on ANN Model

In [33]:
def crossval_model():
    classifier=Sequential()
    classifier.add(Dense(input_dim=11,units=6,kernel_initializer='uniform',activation='relu'))
    classifier.add(Dense(units=6,kernel_initializer='uniform',activation='relu'))
    classifier.add(Dense(units=1,kernel_initializer='uniform',activation='sigmoid'))
    classifier.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return classifier

In [34]:
from scikeras.wrappers import KerasClassifier
classifier=KerasClassifier(crossval_model,batch_size=32,epochs=10)

In [35]:
from sklearn.model_selection import cross_val_score
cv_score=(cross_val_score(classifier,X,y,cv=2))
cv_score=cv_score.mean()

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7628 - loss: 0.7358
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8041 - loss: 0.5253
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7937 - loss: 0.5332
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8010 - loss: 0.5114
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7919 - loss: 0.5208
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7912 - loss: 0.5184
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7939 - loss: 0.5113
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8038 - loss: 0.4903
Epoch 9/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8063 - loss: 0.4905
Epoch 10/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7889 - loss: 0.5121
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step
Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7605 - loss: 0.7080
Epoch 2/10

In [36]:
cv_score

0.7963

In [37]:
estimator=KerasClassifier(crossval_model())
param_grid={"batch_size":[32],"epochs":[10,20],"optimizer":['adam','rmsprop']}

In [38]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(estimator,param_grid,scoring='accuracy',n_jobs=-1)
grid_result=grid.fit(X_train,y_train)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7895 - loss: 0.6422
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8079 - loss: 0.4205
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7924 - loss: 0.4261
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7952 - loss: 0.4143
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7944 - loss: 0.4166
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7907 - loss: 0.4031
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8222 - loss: 0.4027
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8284 - loss: 0.3820
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8215 - loss: 0.3953
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8263 - loss: 0.3875
Epoch 11/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8336 - loss: 0.3757
Epoch 12/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

In [39]:
grid_result.best_params_

{'batch_size': 32, 'epochs': 20, 'optimizer': 'adam'}

In [41]:
final_ann_model= grid_result.best_estimator_

In [44]:
grid_result.best_score_

0.85

In [46]:
ypred_test=final_ann_model.predict(X_test)
ypred_test=ypred_test>0.5.astype(int)  # Ensure it's 0 or 1

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [47]:
print(f"Final model accuracy score is {accuracy_score(y_test,ypred_test)}")

confusion_matrix(y_test,ypred_test)

Final model accuracy score is 0.856


array([[1561,   34],
       [ 254,  151]], dtype=int64)

## ✅ Conclusion

The neural network-based churn prediction model successfully identified patterns in customer behavior using demographic and financial data.  
With an accuracy of **85.6%**, the model proved effective in distinguishing between customers likely to churn and those likely to stay.  
Key features such as **credit score**, **age**, and **account activity** significantly influenced churn prediction, providing valuable insights.  
This solution empowers the bank to implement **targeted retention strategies**, reduce customer loss, and enhance long-term profitability.
